# 미니 프로젝트 템플릿 ② — 텍스트 분류 (18_project_text_template)

11주차 `16_text_finetune` 을 **내 프로젝트용으로 넓힌 템플릿**입니다. 빈칸은 없습니다.
사전학습 한국어 모델 **KoELECTRA-small** 을 내 문장 데이터에 맞게 파인튜닝합니다.

```
  셀 0  설정 (CSV 파일 · 모델 · 문장 최대 길이)        ← 내 프로젝트에 맞게 고친다
  셀 1  데이터 불러오기 + 훈련/검증 나누기 + 클래스별 개수 점검
  셀 2  토크나이저 확인 (문장이 어떻게 쪼개지나)
  셀 3  도구 함수 (모델 만들기 · 학습 · 실험 기록)      ← 고치지 않는다
  셀 4  E0 기준선 — 전체 미세조정
  셀 5~7  E1·E2·E3 개선 실험 슬롯                     ← 인자만 바꾼다
  셀 8  실험 기록표 (results/experiments.csv)
  셀 9  가장 좋은 실험 평가 — 혼동행렬 · 클래스별 성적
  셀 10 틀린 문장 보기 (오분류 분석)
  셀 11 내 문장 예측
  셀 12 보고서용 요약
```

### 내 데이터 형식 — `my_text.csv`

```
text,label
"배송이 빠르고 포장도 꼼꼼해요",긍정
"사이즈가 너무 작아서 반품했습니다",부정
```

- 첫 줄은 반드시 `text,label`. 라벨은 한글·영어 아무거나 (예: 긍정/부정, 스포츠/경제/연예)
- 엑셀에서 만들었다면 **"CSV UTF-8"** 로 저장
- 파일이 없으면 연습용 `sample_text.csv` (영화 리뷰 긍정·부정 1,000개)로 실행됩니다
- 실험 결과는 **`results/experiments.csv`** 에 한 줄씩 쌓입니다 (이미지 템플릿과 같은 열)

### 실험 기록표의 열

| 열 | 뜻 |
|---|---|
| `exp_id` | 실험 번호 (E0 = 기준선, E1·E2·E3 … = 개선 실험) |
| `timestamp` | 실행한 날짜·시각 |
| `task` | `image` 또는 `text` |
| `method` | 모델 방식 (예: `resnet_frozen`, `finetune`) |
| `change` | **기준선에서 무엇을 바꿨는지** 한 줄 (내가 적는 칸) |
| `setting` | 그 밖의 설정 (증강 여부, 문장 최대 길이 등) |
| `epochs / lr / batch_size` | 학습 설정 |
| `trainable_params` | 학습한 파라미터 수 |
| `best_val_acc / best_epoch` | 최고 검증 정확도와 그때의 epoch |
| `final_train_loss / final_val_loss` | 마지막 epoch 의 훈련·검증 손실 (차이가 크면 과적합 의심) |
| `train_sec` | 학습 시간(초) |
| `model_path` | 최고 시점 가중치 파일 |

In [ ]:
# 셀 0 — ★ 설정 : 내 프로젝트에 맞게 이 셀만 고친다
PROJECT    = "내 텍스트 분류 프로젝트"
DATA_CSV   = "my_text.csv"                  # text,label 두 열짜리 CSV
MODEL_NAME = "monologg/koelectra-small-v3-discriminator"   # 11주차와 같은 한국어 사전학습 모델
MAX_LEN    = 64                             # 문장 최대 토큰 수 (긴 글이면 128)
BATCH      = 32                             # GPU 메모리가 부족하다는 오류가 나면 16
VAL_RATIO  = 0.2                            # 검증용으로 떼어 둘 비율
SEED       = 42
CPU_MAX_TRAIN = 800                         # CPU 에서는 훈련 문장을 이만큼만 (GPU 면 전부)

In [ ]:
# 셀 1 — 준비와 데이터 불러오기
import os, csv, time, random, datetime
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.utils import logging as hf_logging
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

hf_logging.set_verbosity_error()                 # 모델을 불러올 때의 긴 안내문 숨기기
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
os.makedirs("models", exist_ok=True); os.makedirs("results", exist_ok=True)

def set_seed(s=SEED):
    random.seed(s); torch.manual_seed(s)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("장치 :", device)

if not os.path.exists(DATA_CSV):
    print(f"'{DATA_CSV}' 가 없어 연습용 데이터로 실행합니다.")
    from sample_text import make_sample_text
    DATA_CSV = make_sample_text()

with open(DATA_CSV, encoding="utf-8-sig") as f:
    data = [(r["text"].strip(), r["label"].strip()) for r in csv.DictReader(f) if r["text"].strip()]
CLASSES = sorted({lab for _, lab in data})
NUM_CLASSES = len(CLASSES)
L2I = {c: i for i, c in enumerate(CLASSES)}

# 클래스마다 같은 비율로 검증용을 떼어 낸다 (층화 분할)
rng = random.Random(SEED)
train_data, val_data = [], []
for c in CLASSES:
    items = [d for d in data if d[1] == c]; rng.shuffle(items)
    k = max(1, int(len(items) * VAL_RATIO))
    val_data += items[:k]; train_data += items[k:]
rng.shuffle(train_data)
if device.type == "cpu" and len(train_data) > CPU_MAX_TRAIN:
    train_data = train_data[:CPU_MAX_TRAIN]
    print(f"CPU 라서 훈련 문장을 {CPU_MAX_TRAIN}개로 줄였습니다 (GPU 면 전부 사용)")

print("데이터 파일 :", DATA_CSV)
print(f"클래스 {NUM_CLASSES}개")
for c in CLASSES:
    print(f"   {c:<10} 훈련 {sum(1 for _, l in train_data if l == c):5d}개 / 검증 {sum(1 for _, l in val_data if l == c):4d}개")
lens = sorted(len(t) for t, _ in data)
print(f"문장 길이(글자) : 중간값 {lens[len(lens)//2]} · 최대 {lens[-1]}")

In [ ]:
# 셀 2 — 토크나이저 확인 : 문장이 어떤 조각(토큰)으로 쪼개지나
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
for t, lab in train_data[:3]:
    toks = tokenizer.tokenize(t)
    print(f"[{lab}] {t[:40]}")
    print(f"      → {len(toks)}개 토큰 : {toks[:12]}{' …' if len(toks) > 12 else ''}")
n_tok = sorted(len(tokenizer.tokenize(t)) for t, _ in train_data)
over = sum(1 for n in n_tok if n > MAX_LEN - 2)
print(f"\n토큰 수 중간값 {n_tok[len(n_tok)//2]} · MAX_LEN({MAX_LEN})보다 길어 잘리는 문장 {over}개 ({over/len(n_tok)*100:.1f}%)")

---

## 셀 3 — 도구 함수 (고치지 않습니다)

| 함수 | 하는 일 |
|---|---|
| `build_model(method)` | `"finetune"` (전체 미세조정) · `"frozen"` (본체 동결, 분류 머리만 학습) |
| `run_experiment(...)` | 학습 → 최고 시점 저장 → 곡선 → **실험 기록표에 한 줄 추가** |
| `show_log()` | 지금까지의 실험 기록표 보기 |

In [ ]:
# 셀 3 — 도구 함수
def make_loader(items, max_len, shuffle):
    def collate(batch):
        enc = tokenizer([t for t, _ in batch], padding=True, truncation=True,
                        max_length=max_len, return_tensors="pt")
        enc["labels"] = torch.tensor([L2I[l] for _, l in batch])
        return enc
    return DataLoader(items, batch_size=BATCH, shuffle=shuffle, collate_fn=collate)

def build_model(method):
    set_seed()
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_CLASSES)
    if method == "frozen":
        for p in m.base_model.parameters():
            p.requires_grad = False               # 본체(사전학습 부분) 동결 → 분류 머리만 학습
    elif method != "finetune":
        raise ValueError(f"모르는 방식입니다 : {method}")
    return m.to(device)

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); total = correct = n = 0
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        out = model(**b)
        total += out.loss.item() * len(b["labels"])
        correct += (out.logits.argmax(1) == b["labels"]).sum().item(); n += len(b["labels"])
    return total / n, correct / n

LOG_PATH = "results/experiments.csv"
LOG_COLS = ['exp_id', 'timestamp', 'task', 'method', 'change', 'setting', 'epochs', 'lr', 'batch_size', 'trainable_params', 'best_val_acc', 'best_epoch', 'final_train_loss', 'final_val_loss', 'train_sec', 'model_path']

def log_experiment(row):
    """실험 한 번의 결과를 results/experiments.csv 에 한 줄 덧붙인다 (지우지 않고 계속 쌓인다)."""
    new = not os.path.exists(LOG_PATH)
    with open(LOG_PATH, "a", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=LOG_COLS)
        if new:
            w.writeheader()
        w.writerow({k: row.get(k, "") for k in LOG_COLS})

def read_log():
    if not os.path.exists(LOG_PATH):
        return []
    with open(LOG_PATH, encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))

def show_log():
    rows = read_log()
    if not rows:
        print("아직 기록된 실험이 없습니다."); return
    print(f"{'번호':<5}{'방식':<17}{'최고 검증 정확도':>16}{'epoch':>7}{'학습 파라미터':>15}{'시간(초)':>10}   바꾼 것")
    print("-" * 100)
    for r in rows:
        print(f"{r['exp_id']:<5}{r['method']:<17}{float(r['best_val_acc'])*100:>15.1f}%{r['best_epoch']:>7}"
              f"{int(r['trainable_params']):>15,}{float(r['train_sec']):>10.1f}   {r['change']}")
    best = max(rows, key=lambda r: float(r["best_val_acc"]))
    print(f"\n가장 좋은 실험 : {best['exp_id']} ({best['change']}) — {float(best['best_val_acc'])*100:.1f}%")

def plot_history(h, title):
    ep = range(1, len(h["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    ax[0].plot(ep, h["train_loss"], "o-", label="훈련 손실"); ax[0].plot(ep, h["val_loss"], "o-", label="검증 손실")
    ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title(f"{title} — 손실")
    ax[1].plot(ep, [a * 100 for a in h["val_acc"]], "o-", color="tab:green")
    ax[1].set_xlabel("epoch"); ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].set_title(f"{title} — 검증 정확도")
    plt.tight_layout(); plt.savefig(f"results/{title.split()[0]}_curve.png", dpi=110); plt.show()

def run_experiment(exp_id, change, method="finetune", epochs=3, lr=5e-5, max_len=None):
    """실험 한 번 = 학습 + 최고 시점 저장 + 곡선 + 기록표 한 줄"""
    max_len = max_len or MAX_LEN
    model = build_model(method)
    train_loader = make_loader(train_data, max_len, shuffle=True)
    val_loader = make_loader(val_data, max_len, shuffle=False)
    set_seed()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    h = {"train_loss": [], "val_loss": [], "val_acc": []}
    best, best_ep, path, t0 = -1.0, 0, f"models/{exp_id}.pt", time.time()
    for ep in range(1, epochs + 1):
        model.train(); run = n = 0
        for b in train_loader:
            b = {k: v.to(device) for k, v in b.items()}
            loss = model(**b).loss                # 정답(labels)을 함께 넣으면 손실까지 계산해 준다
            opt.zero_grad(); loss.backward(); opt.step()
            run += loss.item() * len(b["labels"]); n += len(b["labels"])
        vl, va = evaluate(model, val_loader)
        h["train_loss"].append(run / n); h["val_loss"].append(vl); h["val_acc"].append(va)
        mark = ""
        if va > best:
            best, best_ep = va, ep
            torch.save(model.state_dict(), path); mark = "  ← 최고, 저장"
        print(f"[{exp_id}] epoch {ep:2d} | 훈련 손실 {run/n:.4f} | 검증 손실 {vl:.4f} | 검증 정확도 {va*100:5.1f}%{mark}")
    sec = time.time() - t0
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{exp_id}] 최고 검증 정확도 {best*100:.1f}% (epoch {best_ep}) | 학습 파라미터 {trainable:,} | {sec:.1f}초")
    log_experiment(dict(exp_id=exp_id, timestamp=datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
                        task="text", method=method, change=change, setting=f"max_len={max_len}, n_train={len(train_data)}",
                        epochs=epochs, lr=lr, batch_size=BATCH, trainable_params=trainable,
                        best_val_acc=round(best, 4), best_epoch=best_ep,
                        final_train_loss=round(h["train_loss"][-1], 4), final_val_loss=round(h["val_loss"][-1], 4),
                        train_sec=round(sec, 1), model_path=path))
    plot_history(h, f"{exp_id} {change}")
    return h

print("도구 준비 완료 : build_model / run_experiment / show_log")

---

## 셀 4 — E0 기준선

11주차와 같은 방식(**전체 미세조정**, 학습률 `5e-5`, 3 epoch)을 기준선으로 둡니다.

In [ ]:
# 셀 4 — E0 기준선 : KoELECTRA-small 전체 미세조정 (11주차 방식)
h0 = run_experiment("E0", "기준선 (전체 미세조정)", method="finetune", epochs=3, lr=5e-5)

---

## 셀 5~7 — 개선 실험 슬롯

**한 번에 하나만 바꿉니다.** `change` 에 무엇을 바꿨는지 한 줄로 적습니다.

| 바꿔 볼 것 | 인자 | 예시 |
|---|---|---|
| 학습률 | `lr` | `5e-5` → `2e-5` 또는 `1e-4` |
| epoch 수 | `epochs` | 3 → 5 (검증 손실이 아직 내려가고 있다면) |
| 동결 vs 미세조정 | `method` | `"frozen"` (분류 머리만 — 학습률은 크게 `1e-3`) |
| 문장 최대 길이 | `max_len` | 64 → 128 (셀 2 에서 잘리는 문장이 많다면) |
| 데이터 | CSV | 문장 추가 · 라벨이 애매한 문장 정리 (가장 효과가 큰 경우가 많다) |

아래 세 슬롯은 **예시 값**입니다. 내 프로젝트에 맞게 바꿔 쓰세요.

In [ ]:
# 셀 5 — E1 : (예시) 학습률을 두 배로
h1 = run_experiment("E1", "학습률 1e-4", method="finetune", epochs=3, lr=1e-4)

In [ ]:
# 셀 6 — E2 : (예시) 본체를 얼리고 분류 머리만 학습하면?
h2 = run_experiment("E2", "본체 동결 (머리만)", method="frozen", epochs=3, lr=1e-3)

In [ ]:
# 셀 7 — E3 : (예시) 더 오래 학습하면?
h3 = run_experiment("E3", "epoch 5", method="finetune", epochs=5, lr=5e-5)

---

## 셀 8 — 실험 기록표

`results/experiments.csv` 는 **엑셀로도 열립니다.** 같은 번호로 다시 실행하면 새 줄이 추가됩니다.

In [ ]:
# 셀 8 — 실험 기록표 보기
show_log()

In [ ]:
# 셀 9 — 가장 좋은 실험 : 혼동행렬 + 클래스별 성적
BEST_ID = ""                                   # 비우면 검증 정확도가 가장 높은 실험

rows = read_log()
pick = [r for r in rows if r["exp_id"] == BEST_ID] if BEST_ID else rows
best_row = max(pick, key=lambda r: float(r["best_val_acc"]))
best_model = build_model(best_row["method"])
best_model.load_state_dict(torch.load(best_row["model_path"], map_location=device))
print(f"평가할 실험 : {best_row['exp_id']} ({best_row['change']})")

val_loader = make_loader(val_data, MAX_LEN, shuffle=False)

@torch.no_grad()
def predict_all(model):
    model.eval(); ys, ps = [], []
    for b in val_loader:
        b = {k: v.to(device) for k, v in b.items()}
        ps += model(**b).logits.argmax(1).cpu().tolist(); ys += b["labels"].cpu().tolist()
    return ys, ps

ys, ps = predict_all(best_model)
fig, ax = plt.subplots(figsize=(1.3 * NUM_CLASSES + 2.5, 1.3 * NUM_CLASSES + 2))
ConfusionMatrixDisplay(confusion_matrix(ys, ps), display_labels=CLASSES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"{best_row['exp_id']} 혼동행렬"); ax.set_xlabel("모델의 예측"); ax.set_ylabel("정답")
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=120); plt.show()
print(classification_report(ys, ps, target_names=CLASSES, digits=3, zero_division=0))

In [ ]:
# 셀 10 — 틀린 문장 보기 : 사람이 봐도 애매한 문장인가? 반어법인가? 라벨이 틀렸나?
wrong = [(val_data[i][0], CLASSES[y], CLASSES[p]) for i, (y, p) in enumerate(zip(ys, ps)) if y != p]
print(f"검증 {len(ys)}개 중 틀린 문장 {len(wrong)}개\n")
for t, y, p in wrong[:10]:
    print(f"  정답 {y:<5} 예측 {p:<5} | {t[:60]}")
with open("results/wrong_examples.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f); w.writerow(["text", "정답", "예측"]); w.writerows(wrong)
print("\n전체 목록 : results/wrong_examples.csv")

In [ ]:
# 셀 11 — 내 문장 예측 (학습에 쓰지 않은 문장)
MY_SENTENCES = [
    "생각보다 훨씬 좋았어요. 또 보고 싶네요",
    "기대했는데 시간만 아까웠다",
]

@torch.no_grad()
def predict_text(model, sentences):
    model.eval()
    enc = tokenizer(sentences, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    prob = torch.softmax(model(**enc).logits, dim=1).cpu()
    for s, p in zip(sentences, prob):
        print(f"{CLASSES[p.argmax()]} ({p.max()*100:5.1f}%) | {s}")

predict_text(best_model, MY_SENTENCES)

In [ ]:
# 셀 12 — 보고서용 요약 (그대로 복사해 보고서 초안에 붙인다)
rows = read_log()
base = next((r for r in rows if r["exp_id"] == "E0"), rows[0])
print(f"■ 프로젝트 : {PROJECT}")
print(f"■ 데이터   : 클래스 {NUM_CLASSES}개 {CLASSES} · 훈련 {len(train_data)}개 · 검증 {len(val_data)}개")
print(f"■ 기준선   : {base['exp_id']} {base['change']} → 검증 정확도 {float(base['best_val_acc'])*100:.1f}%")
print(f"■ 최고     : {best_row['exp_id']} {best_row['change']} → 검증 정확도 {float(best_row['best_val_acc'])*100:.1f}%"
      f" (기준선 대비 {(float(best_row['best_val_acc'])-float(base['best_val_acc']))*100:+.1f}%p)")
print(f"■ 실험 수  : {len(rows)}회  (results/experiments.csv)")
print("■ 파일     : results/confusion_matrix.png · results/wrong_examples.csv · results/E*_curve.png")